# 02. Очистка данных и формирование признаков

Ноутбук содержит операции, выполненные поверх авторской подготовки: фильтрацию выборки, обработку выбросов, нормировку рейтинга и расчёт производных признаков для моделирования.

Последовательность операций: контроль дубликатов, отбор когорты, обработка типов оплаты, обработка выбросов в рейтинге, нормировка, расчёт целевых переменных, сохранение результирующих таблиц.

In [1]:
import sqlite3
from contextlib import closing

import numpy as np
import pandas as pd

DB_PATH = '../data/churn.db'
OBSERVATION_CUTOFF = pd.Timestamp('2026-03-01')
COHORT_YEAR = 2024
MAX_GROUP_SIZE = 14

DATE_COLUMNS = [
    'first_course_start_date',
    'first_course_end_date',
    'group_churn_date',
    'second_course_payment_date',
]


def load_table(name):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        frame = pd.read_sql_query(f'SELECT * FROM {name}', conn)
    for column in [c for c in DATE_COLUMNS if c in frame.columns]:
        frame[column] = pd.to_datetime(frame[column])
    return frame


df = load_table('students_prepared')
print(f'Загружено: {len(df):,} строк')
df.head()

Загружено: 7,295 строк


,id,gender,utm_source,age,payment_course,first_course_name,course_format,first_payment_type,group_id,first_course_start_date,...,missed_classes,incomplete_hw,rating,cs_tickets,second_course_payment_date,course_clean,course_group,utm_source_group,course_format_group,payment_type_group
0,36056830,UNKNOWN,flocktory,13,Unity,[858]3D игры на Unity[None][13-17][90 min][32 ...,Индивидуальная группа,Full,28733,2024-01-20,...,0.038462,0.925926,1,11,2024-03-17,3d игры на unity,unity,referral,individual,full
1,36262276,WOMAN,ig,12,Цифровой дизайн,"[610] Дизайн цифровых миров [2022][10-12][50m,...",Микро-группа,Full,30350,2024-03-28,...,0.058824,0.909091,3,18,2024-11-09,дизайн цифровых миров,animation,meta,premium,full
2,36288588,UNKNOWN,ig,13,Python,[721] Python Base Internship LVL 1 [2022][13+]...,Стандартная группа,Full,30591,2024-04-07,...,0.000000,0.666667,2,14,NaT,python base internship lvl 1,python1,meta,standard,full
3,36812572,MAN,website,15,[887] Python LVL 2 [2023][13+][90min][32L][Ru]...,[887]Python Pro[None][13-17][90 min][32 L][CIS...,Premium Group,Internal installment,35068,2024-08-11,...,0.093750,0.406250,3,28,NaT,python pro,pythonPRO,website,premium,internal_installment
4,36355456,UNKNOWN,flocktory,7,FunTech Explorers,[1053]FunTech Explorers[2023][5-7][RU],Мини-группа | Короткая,Bank installment,30904,2024-04-15,...,0.100000,0.000000,1,12,NaT,funtech explorers,funtech,referral,standard,bank_installment


## Контроль дубликатов

Уникальность ключа в исходной выгрузке не проверялась, вследствие чего проверка выполняется на данном этапе. Выявлена одна пара наблюдений с идентичными идентификатором, датой начала обучения и форматом, но различными значениями рейтинга, что соответствует повторному попаданию одного ученика в выгрузку.

Сохраняется наблюдение с наилучшим значением рейтинга. Случай единичный и на результаты анализа не влияет, однако проверку целесообразно сохранить в составе процедуры подготовки данных.

In [2]:
duplicated_ids = df['id'].duplicated(keep=False)
print(f'Строк с повторяющимся id: {duplicated_ids.sum()}')
print(f'Полных дублей строки: {df.duplicated().sum()}')

if duplicated_ids.any():
    display(df.loc[duplicated_ids, ['id', 'course_format', 'first_course_start_date', 'rating',
                                    'second_course_payment_date']])

df = df.sort_values('rating').drop_duplicates('id', keep='first').sort_index()
print(f'После удаления дублей: {len(df):,} строк')

Строк с повторяющимся id: 2
Полных дублей строки: 0


,id,course_format,first_course_start_date,rating,second_course_payment_date
2675,34087050,Стандартная группа,2024-02-24,8,NaT
5722,34087050,Стандартная группа,2024-02-24,2,NaT


После удаления дублей: 7,294 строк


## Отбор когорты

Выборка формировалась как совокупность учеников, впервые начавших обучение в 2024 году, однако в выгрузку вошли и иные периоды: пять наблюдений 2023 года, 354 наблюдения 2025 года и одно наблюдение 2026 года.

В анализе используется только когорта 2024 года. Это обеспечивает однородность выборки и сопоставимую длительность наблюдения: горизонт данных ограничен 1 марта 2026 года, вследствие чего даже для декабрьской когорты период наблюдения превышает четырнадцать месяцев.

In [3]:
start_year = df['first_course_start_date'].dt.year
display(start_year.value_counts().sort_index().to_frame('студентов'))

df = df[start_year == COHORT_YEAR].copy()
print(f'Оставлена когорта {COHORT_YEAR}: {len(df):,} строк')

,студентов
first_course_start_date,
2023,5
2024,6934
2025,354
2026,1


Оставлена когорта 2024: 6,934 строк


## Обработка предоплаты

Значение `Downpayment` соответствует внесению задатка, а не полной оплате курса: основной платёж вносится отдельной транзакцией. Следовательно, для соответствующих наблюдений тип первой оплаты зафиксирован некорректно.

Проверяется возможность восстановления основного платежа: при его наличии в выгрузке идентификатор ученика встречался бы повторно. Совпадений не выявлено, единица наблюдения — одна строка на ученика, восстановление фактического типа оплаты невозможно. Соответствующие 86 наблюдений исключаются из выборки.

In [4]:
downpayment = df['first_payment_type'] == 'Downpayment'
downpayment_ids = set(df.loc[downpayment, 'id'])

print(f'Строк с предоплатой: {downpayment.sum()}')
print(f'Из них id, встречающихся в выборке ещё раз: '
      f'{df[df["id"].isin(downpayment_ids)].shape[0] - downpayment.sum()}')

df = df[~downpayment].copy()
print(f'После удаления предоплаты: {len(df):,} строк')

Строк с предоплатой: 86
Из них id, встречающихся в выборке ещё раз: 0
После удаления предоплаты: 6,848 строк


## Обработка выбросов в рейтинге

Показатель `rating` отражает место ученика в группе по успеваемости, где единица соответствует наилучшему результату. Максимальная численность учебной группы составляет 14 человек, следовательно, ранг выше 14 физически недостижим.

Выявлено 280 таких наблюдений, распределённых неравномерно: некорректные значения присутствуют в 167 группах из 1865, при этом общая численность учеников в этих группах составляет 1468 человек. Исключение групп целиком привело бы к потере более чем пятой части выборки.

Принято компромиссное решение: 280 наблюдений с недостижимыми рангами исключаются, группы сохраняются, знаменатель нормировки ограничивается значением 14. Признак `rating_group_broken` фиксирует группы с некорректным рейтингом, что позволяет в ноутбуке 04 проверить устойчивость результатов на подвыборке без них.

In [5]:
group_stats = df.groupby('group_id')['rating'].agg(students_in_sample='size', max_rating='max')
broken_groups = group_stats.index[group_stats['max_rating'] > MAX_GROUP_SIZE]

rating_audit = pd.Series({
    'строк с рангом выше максимума группы': int((df['rating'] > MAX_GROUP_SIZE).sum()),
    'групп с невозможным максимальным рангом': len(broken_groups),
    'всего групп': len(group_stats),
    'студентов в таких группах': int(df['group_id'].isin(broken_groups).sum()),
}, name='значение').to_frame()

display(rating_audit)
display(df.loc[df['rating'] >= MAX_GROUP_SIZE - 1, 'rating'].value_counts().sort_index().to_frame('студентов'))

df['rating_group_broken'] = df['group_id'].isin(broken_groups)
df = df[df['rating'] <= MAX_GROUP_SIZE].copy()
print(f'После удаления невозможных рангов: {len(df):,} строк')

,значение
строк с рангом выше максимума группы,280
групп с невозможным максимальным рангом,167
всего групп,1865
студентов в таких группах,1468


,студентов
rating,
13,165
14,118
15,118
16,65
17,52
18,21
19,15
20,5
21,1


После удаления невозможных рангов: 6,568 строк


## Нормировка рейтинга

Исходный ранг несопоставим между форматами обучения: пятое место в группе из четырнадцати человек и пятое место в группе из пяти отражают противоположные результаты. Для сопоставимости требуется знаменатель — численность группы.

В явном виде численность в данных отсутствует, однако доступны две оценки снизу: количество учеников группы, попавших в выборку, и максимальный ранг внутри группы. Используется максимум из двух оценок, поскольку выборка ограничена учениками 2024 года и фактическая численность групп превышает наблюдаемую. Сверху значение ограничивается максимальной численностью группы.

Формула нормировки: `rating_norm = (rating − 1) / (численность группы − 1)`, где нулю соответствует наилучший результат в группе, единице — наихудший.

Результат представлен в таблице: медианное значение нормированного рейтинга составляет 0.5 во всех трёх форматах, что подтверждает сопоставимость шкалы. Одновременно фиксируется ограничение индивидуального формата: для 651 ученика из 653 рейтинг не определён, поскольку группа состоит из одного человека.

In [6]:
group_denominator = (
    df.groupby('group_id')['rating']
    .agg(students_in_sample='size', max_rating='max')
    .max(axis=1)
    .clip(upper=MAX_GROUP_SIZE)
    .rename('group_size')
)

df = df.join(group_denominator, on='group_id')
df['rating_norm'] = np.where(
    df['group_size'] > 1,
    (df['rating'] - 1) / (df['group_size'] - 1),
    np.nan,
)

display(
    df.groupby('course_format_group', observed=True)
    .agg(students=('id', 'size'),
         median_group_size=('group_size', 'median'),
         median_rating=('rating', 'median'),
         median_rating_norm=('rating_norm', 'median'),
         undefined_rating=('rating_norm', lambda s: s.isna().sum()))
    .round(3)
)

,students,median_group_size,median_rating,median_rating_norm,undefined_rating
course_format_group,,,,,
individual,653,1.0,1.0,0.5,651
premium,3406,7.0,4.0,0.5,12
standard,2509,13.0,7.0,0.5,3


## Целевые переменные и производные признаки

Определяются два события, составляющие основу дальнейшего анализа:

- **выбытие с первого курса** — дата в поле `group_churn_date`, заполнена более чем для половины когорты и во всех случаях предшествует завершению курса;
- **приобретение второго курса** — дата в поле `second_course_payment_date`.

Дополнительно рассчитываются временные характеристики: длительность курса, период наблюдения от начала обучения до границы горизонта данных, время до наступления каждого события.

Отдельно формируется индикатор `bought_before_course_end`, фиксирующий приобретение второго курса до завершения первого. Признак используется при моделировании: 76% повторных покупок совершаются до окончания обучения, что определяет постановку задачи прогнозирования.

In [7]:
df['course_duration_days'] = (df['first_course_end_date'] - df['first_course_start_date']).dt.days
df['observed_days'] = (OBSERVATION_CUTOFF - df['first_course_start_date']).dt.days

df['dropped_out'] = df['group_churn_date'].notna()
df['days_to_dropout'] = (df['group_churn_date'] - df['first_course_start_date']).dt.days

df['bought_second'] = df['second_course_payment_date'].notna()
df['days_to_second'] = (df['second_course_payment_date'] - df['first_course_start_date']).dt.days
df['days_after_course_end'] = (df['second_course_payment_date'] - df['first_course_end_date']).dt.days

df['bought_before_course_end'] = df['bought_second'] & (df['days_after_course_end'] <= 0)

summary = pd.Series({
    'студентов': len(df),
    'ушли из группы до конца курса': int(df['dropped_out'].sum()),
    'купили второй курс': int(df['bought_second'].sum()),
    'из них до окончания первого курса': int(df['bought_before_course_end'].sum()),
    'медиана длительности курса, дней': int(df['course_duration_days'].median()),
    'минимум наблюдения после старта, дней': int(df['observed_days'].min()),
}, name='значение').to_frame()

display(summary)

,значение
студентов,6568
ушли из группы до конца курса,3444
купили второй курс,2118
из них до окончания первого курса,1610
"медиана длительности курса, дней",224
"минимум наблюдения после старта, дней",428


## Формирование набора для моделирования

Индивидуальный формат обучения исключается из моделирования. Основание — свойства показателя рейтинга.

При индивидуальном обучении ранг ученика тождественно равен единице, что обусловлено численностью группы, а не результатами обучения. Таким образом, показатель не измеряет успеваемость. Сохранение таких наблюдений в обучающей выборке приводит к одному из двух нежелательных исходов: коэффициент при рейтинге утрачивает содержательную интерпретацию и начинает отражать формат обучения, либо модель фиксирует зависимость между единичным рангом и повторной покупкой и распространяет её на успевающих учеников групповых форматов без достаточных оснований.

При этом рейтинг является наиболее значимым поведенческим предиктором в данных, вследствие чего исключение 653 наблюдений предпочтительнее отказа от признака. Для индивидуального формата требуется отдельная модель на ином наборе признаков.

В таблице `students_clean` наблюдения индивидуального формата сохраняются: расчёт продуктовых метрик в ноутбуке 03 выполняется по всей когорте.

In [8]:
model_input = df[df['course_format_group'] != 'individual'].copy()

print(f'Исключено индивидуальное обучение: {len(df) - len(model_input):,} студентов')
print(f'Осталось для моделирования: {len(model_input):,}')
print(f'Не определён нормированный рейтинг: {model_input["rating_norm"].isna().sum()}')

model_input = model_input[model_input['rating_norm'].notna()].copy()
print(f'Итоговый набор: {len(model_input):,} строк, '
      f'купили второй курс {model_input["bought_second"].mean() * 100:.1f}%')

Исключено индивидуальное обучение: 653 студентов
Осталось для моделирования: 5,915
Не определён нормированный рейтинг: 15
Итоговый набор: 5,900 строк, купили второй курс 33.0%


In [9]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    df.to_sql('students_clean', conn, if_exists='replace', index=False)
    model_input.to_sql('model_input', conn, if_exists='replace', index=False)
    conn.commit()

    saved = pd.read_sql_query('''
        SELECT 'students_clean' AS таблица, COUNT(*) AS строк FROM students_clean
        UNION ALL SELECT 'model_input', COUNT(*) FROM model_input;
    ''', conn)

saved

,таблица,строк
0,students_clean,6568
1,model_input,5900


## Результат

Сформированы две таблицы:

- **`students_clean`** — 6 568 наблюдений, когорта 2024 года после очистки. Используется для расчёта продуктовых метрик и построения воронки.
- **`model_input`** — 5 900 наблюдений без индивидуального формата обучения и с определённым значением нормированного рейтинга. Используется для моделирования и анализа выживаемости.

Конверсия во второй курс в наборе для моделирования составляет 33.0%.